In [ ]:
# Kaggle: enable Internet and a GPU, and attach Multi-topology-pcb using Add Input.
# All paths are explicit. No Kaggle input-directory searching is performed.
import os
import sys
import subprocess
import urllib.request
import importlib.util
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

DATA_ROOT = Path(os.environ.get("PCB_DATA_ROOT", "/kaggle/input/datasets/andreimunteanu/multi-topology-pcb"))
WORKING_DIR = Path(os.environ.get("PCB_WORKING_DIR", "/kaggle/working/nn_unet_pcb_sliding_window"))
CODE_DIR = WORKING_DIR / "pcb_code"
CODE_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_TEST = os.environ.get("PCB_SMOKE_TEST", "0") == "1"

# Install tested external packages only when required.
requirements = {"nnunetv2": "2.7.0", "dynamic-network-architectures": "0.4.3"}
needed = []
for name, wanted in requirements.items():
    try:
        if version(name) != wanted:
            needed.append(f"{name}=={wanted}")
    except PackageNotFoundError:
        needed.append(f"{name}=={wanted}")
for package in ("h5py", "pandas", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        needed.append(package)
if needed:
    print("Installing:", needed, flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *needed], check=True)

# Pull only our four small adapter/loader modules, never the dataset.
CODE_REVISION = "main"
CODE_URL = f"https://raw.githubusercontent.com/ongisho/surrogates/{CODE_REVISION}/pcb"
LOCAL_CODE_DIR = os.environ.get("PCB_LOCAL_CODE_DIR")  # explicit override for local integration checks
for name in ("dataset.py", "nn_unet.py", "nn_unet_patch_training.py", "nn_unet_sliding_window.py"):
    print("Loading code:", name, flush=True)
    if LOCAL_CODE_DIR:
        content = (Path(LOCAL_CODE_DIR) / name).read_bytes()
    else:
        with urllib.request.urlopen(f"{CODE_URL}/{name}", timeout=15) as response:
            content = response.read()
    compile(content, name, "exec")
    temporary = (CODE_DIR / name).with_suffix(".py.tmp")
    temporary.write_bytes(content)
    temporary.replace(CODE_DIR / name)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print("Python:", sys.executable)
print("Dataset (explicit path):", DATA_ROOT)
print("Outputs:", WORKING_DIR)


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from pathlib import Path
import json
import time
from dataset import PCBDataset, NormalizationStats
from nn_unet import NNUNetRegressor, PCBImageDataset
from nn_unet_sliding_window import predict_sliding_window
from nn_unet_patch_training import RandomPatchDataset
from importlib.metadata import version
PACKAGE_VERSIONS = {name: version(name) for name in ("nnunetv2", "dynamic-network-architectures")}
print("External packages:", PACKAGE_VERSIONS)

## 1. Hyperparameters

In [ ]:
HPARAMS = {
    # data: official train/validation/test directories
    "data_root": str(DATA_ROOT),
    "train_fraction": 0.05,  # reproducible 5% training subset; validation stays complete
    "run_test": False,  # skip held-out test discovery, evaluation and plots
    "stride": 1,  # native 256 x 256 (higher GPU memory)

    # model: unmodified pip-installed nnU-Net architecture, regression adapter
    "input_channels": 8,  # x, y, power, chip, ceramic, baseplate, copper, cooling
    "out_channels": 1,
    "features_per_stage": [32, 64, 128, 256, 320],
    "n_conv_per_stage": 2,
    "n_conv_per_stage_decoder": 2,

    # training: uniform random cases/crops, with replacement, no augmentation
    "train_patch_size": [64, 64],
    "train_iterations_per_epoch": 20,
    "batch_size": 32,
    "epochs": 30,
    "lr": 1e-3,
    "weight_decay": 0.0,  # no weight decay
    "max_grad_norm": 0.1,

    # sliding-window validation/inference on full boards
    "inference_patch_size": [64, 64],
    "tile_step_size": 0.5,  # at least 50% overlap, with adjusted edge positions
    "sw_batch_size": 4,  # tiles per network call, independent of training batch size
    "use_gaussian": True,

    # runtime
    "num_workers": 0,
    "pin_memory": True,
    "seed": 42,

    # output
    "checkpoint_path": str(WORKING_DIR / "nn_unet_pcb_sliding_window.pt"),
    "last_checkpoint_path": str(WORKING_DIR / "nn_unet_pcb_sliding_window_last.pt"),
    "debug_csv_path": str(WORKING_DIR / "nn_unet_pcb_sliding_window_debug_log.csv"),
}

DEBUG = {
    "stop_on_nonfinite": True,
    "log_every_n_batches": 20,
    "print_every_n_batches": 20,
    "max_train_batches": None,
    "max_eval_batches": None,
    "detect_anomaly": False,
    "save_parameter_snapshots": True,
}

if SMOKE_TEST:
    HPARAMS.update(features_per_stage=[8, 16, 32], epochs=1, batch_size=1,
                   train_iterations_per_epoch=2)
    torch.set_num_threads(2)

torch.autograd.set_detect_anomaly(DEBUG["detect_anomaly"])

## 2. Diagnostics

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def tensor_stats(x):
    x = x.detach()
    finite = torch.isfinite(x)

    out = {
        "finite_fraction": finite.float().mean().item(),
        "nan_count": torch.isnan(x).sum().item(),
        "inf_count": torch.isinf(x).sum().item(),
    }

    if finite.any():
        xf = x[finite].float()
        out.update({
            "mean": xf.mean().item(),
            "std": xf.std(unbiased=False).item(),
            "rms": torch.sqrt(torch.mean(xf * xf)).item(),
            "absmax": xf.abs().max().item(),
            "norm": xf.norm().item(),
            "min": xf.min().item(),
            "max": xf.max().item(),
        })
    else:
        out.update({
            "mean": np.nan, "std": np.nan, "rms": np.nan,
            "absmax": np.nan, "norm": np.nan,
            "min": np.nan, "max": np.nan,
        })

    return out


def check_finite(name, x, context=None):
    if torch.isfinite(x).all():
        return

    context = context or {}
    print("\n" + "=" * 90)
    print("NON-FINITE TENSOR:", name)
    print("context:", context)
    print("shape:", tuple(x.shape))
    print("stats:", tensor_stats(x))
    print("=" * 90)

    if DEBUG["stop_on_nonfinite"]:
        raise FloatingPointError(f"{name} contains NaN/Inf at {context}")


def flatten_finite_tensors(tensors):
    vals = []
    for t in tensors:
        if t is None:
            continue
        a = t.detach().float().reshape(-1).cpu()
        a = a[torch.isfinite(a)]
        if a.numel():
            vals.append(a)
    return torch.cat(vals) if vals else torch.empty(0)


def parameter_group_map(model):
    # Global named_parameters deduplicates upstream decoder/encoder references.
    named = list(model.named_parameters())
    return {
        "input_encoder": [p for name, p in named if name.startswith("network.encoder.stages.0.")],
        "blocks": [p for name, p in named if name.startswith("network.encoder.")
                   and not name.startswith("network.encoder.stages.0.")],
        "decoder": [p for name, p in named if name.startswith("network.decoder.")],
    }


def stats_from_flat(flat):
    if flat.numel() == 0:
        return {
            "mean": np.nan, "std": np.nan, "rms": np.nan,
            "absmax": np.nan, "norm": np.nan,
        }
    return {
        "mean": flat.mean().item(),
        "std": flat.std(unbiased=False).item(),
        "rms": torch.sqrt(torch.mean(flat * flat)).item(),
        "absmax": flat.abs().max().item(),
        "norm": flat.norm().item(),
    }


def group_parameter_stats(params):
    return stats_from_flat(
        flatten_finite_tensors([p.data for p in params])
    )


def group_gradient_stats(params):
    return stats_from_flat(
        flatten_finite_tensors([p.grad for p in params if p.grad is not None])
    )


def snapshot_parameter_values(model):
    return {
        name: flatten_finite_tensors([p.data for p in params]).numpy()
        for name, params in parameter_group_map(model).items()
    }


def snapshot_gradient_values(model):
    return {
        name: flatten_finite_tensors(
            [p.grad for p in params if p.grad is not None]
        ).numpy()
        for name, params in parameter_group_map(model).items()
    }


set_seed(HPARAMS["seed"])

## 3. PCB dataset

In [ ]:
# Read the known metadata file directly. No glob/rglob or directory search.
manifest_path = DATA_ROOT / "metadata" / "manifest.csv"
if not manifest_path.is_file():
    raise FileNotFoundError(
        f"Missing {manifest_path}. Set DATA_ROOT in cell 0 to the exact dataset root "
        "shown in Kaggle's Input sidebar. No automatic directory search is performed."
    )
manifest = pd.read_csv(manifest_path, usecols=["split", "path"])

def indexed_files(split):
    relative_paths = manifest.loc[manifest["split"] == split, "path"]
    files = []
    for value in relative_paths:
        relative = Path(value)
        if relative.is_absolute() or ".." in relative.parts or relative.parts[0] != split:
            raise ValueError(f"Invalid manifest sample path: {value}")
        files.append(DATA_ROOT / relative)
    if not files:
        raise FileNotFoundError(f"No {split} entries in {manifest_path}")
    return sorted(set(files))

print("Reading training file index...", flush=True)
all_train_files = indexed_files("TrainData")
train_fraction = HPARAMS["train_fraction"]
if not 0 < train_fraction <= 1:
    raise ValueError("train_fraction must be in (0, 1]")
n_train = max(1, int(len(all_train_files) * train_fraction))
indices = np.sort(np.random.default_rng(HPARAMS["seed"]).choice(
    len(all_train_files), size=n_train, replace=False,
))
train_files = [all_train_files[i] for i in indices]
print(f"Using {len(train_files)}/{len(all_train_files)} training files "
      f"({100 * train_fraction:g}%, seed={HPARAMS['seed']})", flush=True)
print("Reading validation file index...", flush=True)
val_files = indexed_files("ValData")
test_files = []
if HPARAMS["run_test"]:
    print("Reading test file index...", flush=True)
    test_files = indexed_files("TestData")

# Compute exact statistics from ALL selected training files, then reuse the cache.
# Validation/test targets and unselected training files are not used for fitting.
NORM_MAX_FILES = None
normalizer = NormalizationStats.fit(
    train_files, max_files=NORM_MAX_FILES, seed=HPARAMS["seed"],
    cache_path=WORKING_DIR / "pcb_normalization.json", progress_every=25,
)
train_image_dataset = PCBImageDataset(PCBDataset(HPARAMS["data_root"], "TrainData", normalizer, HPARAMS["stride"], train_files))
train_dataset = RandomPatchDataset(
    train_image_dataset, patch_size=HPARAMS["train_patch_size"],
    samples_per_epoch=HPARAMS["train_iterations_per_epoch"] * HPARAMS["batch_size"],
    seed=HPARAMS["seed"],
)
val_dataset = PCBImageDataset(PCBDataset(HPARAMS["data_root"], "ValData", normalizer, HPARAMS["stride"], val_files))
test_dataset = (PCBImageDataset(PCBDataset(HPARAMS["data_root"], "TestData", normalizer,
                                        HPARAMS["stride"], test_files))
                if HPARAMS["run_test"] else None)

def make_loader(dataset, shuffle=False):
    return DataLoader(dataset, batch_size=HPARAMS["batch_size"], shuffle=shuffle,
                      num_workers=HPARAMS["num_workers"], pin_memory=HPARAMS["pin_memory"])

train_loader = make_loader(train_dataset)  # sampler indices already generate random cases/crops
val_loader = make_loader(val_dataset)
test_loader = make_loader(test_dataset) if HPARAMS["run_test"] else None
print("Training images:", len(train_image_dataset))
print("Training patches per epoch:", len(train_dataset))
print("Training batches per epoch:", len(train_loader))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset) if HPARAMS["run_test"] else "skipped")
print("Training input patch / temperature:", [tuple(a.shape) for a in train_dataset[0]])
print("Validation full image / temperature:", [tuple(a.shape) for a in val_dataset[0]])
print(normalizer)

## 4. Debug recorder

In [ ]:
class DebugRecorder:
    def __init__(self):
        self.rows = []

    def record_tensor(self, tag, x, context):
        row = dict(context)
        row.update({"kind": "tensor", "tag": tag, **tensor_stats(x)})
        self.rows.append(row)

    def record_scalar(self, tag, value, context):
        row = dict(context)
        row.update({"kind": "scalar", "tag": tag, "value": float(value)})
        self.rows.append(row)

    def dataframe(self):
        return pd.DataFrame(self.rows)


recorder = DebugRecorder()

## 5. nnU-Net and diagnostics hooks

In [ ]:
# The U-Net architecture comes from the pip package via nnU-Net's factory.
# Our adapter only reshapes input images and exposes continuous predictions.
diagnostic_context = None
hook_handles = []

def stage_hook(name):
    def hook(module, inputs, output):
        if diagnostic_context is not None:
            check_finite(name, output, diagnostic_context)
            recorder.record_tensor(name, output, diagnostic_context)
    return hook


def predict_field(image):
    return predict_sliding_window(
        model, image, patch_size=HPARAMS["inference_patch_size"],
        tile_step_size=HPARAMS["tile_step_size"], sw_batch_size=HPARAMS["sw_batch_size"],
        use_gaussian=HPARAMS["use_gaussian"],
    )


## 6. Instantiate model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_kwargs = {key: HPARAMS[key] for key in (
    "input_channels", "out_channels", "features_per_stage",
    "n_conv_per_stage", "n_conv_per_stage_decoder")}
model = NNUNetRegressor(**model_kwargs).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=HPARAMS["lr"],
                              weight_decay=HPARAMS["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HPARAMS["epochs"])

for index, stage in enumerate(model.network.encoder.stages):
    hook_handles.append(stage.register_forward_hook(stage_hook(f"encoder.{index}")))
for index, stage in enumerate(model.network.decoder.stages):
    hook_handles.append(stage.register_forward_hook(stage_hook(f"decoder.{index}")))

initial_parameter_snapshot = snapshot_parameter_values(model) if DEBUG["save_parameter_snapshots"] else {}
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print(model)


## 7. Loss and metric

In [ ]:
def relative_l2_per_sample(pred, target):
    B = pred.shape[0]

    pred = pred.reshape(B, -1)
    target = target.reshape(B, -1)

    diff_norm = torch.norm(
        pred - target,
        p=2,
        dim=1,
    )

    target_norm = torch.norm(
        target,
        p=2,
        dim=1,
    )

    return diff_norm / target_norm


def relative_l2_loss(pred, target):
    return relative_l2_per_sample(
        pred,
        target,
    ).sum()


def mae_metric(pred, target):
    return torch.mean(
        torch.abs(pred - target)
    )

class RelativeL2Stats:
    """Exact distribution of the same per-sample RelL2 used by the epoch mean.

    One value per training patch/image or reconstructed validation image.
    Ratios are dimensionless: 0.1 corresponds to 10%, not degrees Celsius.
    """
    def __init__(self):
        self.values = []

    def __enter__(self):
        return self

    def update(self, prediction, target):
        ratios = relative_l2_per_sample(prediction.detach(), target.detach()).float().cpu().numpy()
        if not np.isfinite(ratios).all():
            raise FloatingPointError("Nonfinite per-sample relative L2 errors")
        self.values.append(ratios.copy())

    def result(self):
        if not self.values:
            raise ValueError("No relative L2 errors recorded")
        values = np.concatenate(self.values)
        quantiles = np.percentile(values, [50, 90, 95, 99])
        return dict(zip(
            ("max_rel_l2", "median_rel_l2", "p90_rel_l2", "p95_rel_l2", "p99_rel_l2"),
            [float(values.max()), *map(float, quantiles)],
        ))

    def __exit__(self, *exc):
        self.values.clear()


## 8. Dry run — inspect layer magnitudes before training

In [ ]:
model.eval()
image, target_norm = next(iter(val_loader))
diagnostic_context = {"epoch": 0, "batch": 0, "phase": "dry_run"}
with torch.no_grad():
    pred_norm = predict_field(image)
    pred = normalizer.decode_temperature(pred_norm)
    target = normalizer.decode_temperature(target_norm)
    check_finite("dry_run.prediction", pred, diagnostic_context)
    print("Sliding-window prediction:", tuple(pred.shape), tensor_stats(pred))
    print("Relative L2:", relative_l2_loss(pred, target).item())
    print("MAE (C):", mae_metric(pred, target).item())
diagnostic_context = None
state_rows = recorder.dataframe()
state_rows = state_rows[state_rows["kind"] == "tensor"]
if len(state_rows):
    plt.figure(figsize=(9, 5))
    plt.plot(np.arange(len(state_rows)), state_rows["rms"], marker="o")
    plt.xlabel("Encoder/decoder stage"); plt.ylabel("RMS")
    plt.title("Dry run: layer output RMS across tile batches"); plt.grid(True); plt.tight_layout(); plt.show()

## 9. Training and evaluation with diagnostics

In [ ]:
ERROR_KEYS = ("max_rel_l2", "median_rel_l2", "p90_rel_l2", "p95_rel_l2", "p99_rel_l2")
history = {f"{split}_{metric}": [] for split in ("train", "val")
           for metric in ("rel_l2", "mae", *ERROR_KEYS)}
history["lr"] = []
step_log = []
last_gradient_snapshot = None

def add_group_stats(model, row):
    for group_name, params in parameter_group_map(model).items():
        for prefix, values in (("param", group_parameter_stats(params)),
                               ("grad", group_gradient_stats(params))):
            for key, value in values.items():
                row[f"{prefix}_{group_name}_{key}"] = value

def train_one_epoch_debug(epoch):
    global last_gradient_snapshot, diagnostic_context
    train_dataset.set_epoch(epoch)  # fresh deterministic crops for this epoch
    model.train()
    total_rel_l2 = total_mae = total_samples = 0
    with RelativeL2Stats() as error_stats:
        for batch_idx, batch in enumerate(train_loader):
            if DEBUG["max_train_batches"] is not None and batch_idx >= DEBUG["max_train_batches"]:
                break
            image, target_norm = [a.to(device, non_blocking=True) for a in batch]
            target = normalizer.decode_temperature(target_norm)
            context = {"epoch": epoch, "batch": batch_idx, "phase": "train"}
            detailed = batch_idx % DEBUG["log_every_n_batches"] == 0
            diagnostic_context = context if detailed else None
            optimizer.zero_grad(set_to_none=True)
            pred_norm = model(image)
            diagnostic_context = None
            pred = normalizer.decode_temperature(pred_norm)
            error_stats.update(pred, target)
            loss = relative_l2_loss(pred, target)
            check_finite("train.loss", loss.reshape(1), context)
            loss.backward()
            raw_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), HPARAMS["max_grad_norm"],
                                                           error_if_nonfinite=True)
            row = dict(context, global_optimizer_step=len(step_log), loss_rel_l2=loss.item(),
                       mae=mae_metric(pred, target).item(), raw_grad_norm=raw_grad_norm.item(),
                       pred_absmax=pred.detach().abs().max().item())
            if detailed:
                add_group_stats(model, row)
                if DEBUG["save_parameter_snapshots"]:
                    last_gradient_snapshot = snapshot_gradient_values(model)
            step_log.append(row)
            optimizer.step()
            for name, parameter in model.named_parameters():
                check_finite("parameter." + name, parameter, context)
            batch_size = image.shape[0]
            total_rel_l2 += relative_l2_per_sample(pred.detach(), target).sum().item()
            total_mae += row["mae"] * batch_size
            total_samples += batch_size
            if batch_idx % DEBUG["print_every_n_batches"] == 0:
                print(f"epoch={epoch:03d} batch={batch_idx:04d} loss={row['loss_rel_l2']:.6g} "
                      f"grad={row['raw_grad_norm']:.3g}")
        if not total_samples:
            raise ValueError("No training batches selected")
        metrics = {"rel_l2": total_rel_l2 / total_samples, "mae": total_mae / total_samples}
        metrics.update(error_stats.result())
        return metrics

@torch.no_grad()
def evaluate_debug(loader):
    model.eval()
    total_rel_l2 = total_mae = total_samples = 0
    with RelativeL2Stats() as error_stats:
        for batch_idx, batch in enumerate(loader):
            if DEBUG["max_eval_batches"] is not None and batch_idx >= DEBUG["max_eval_batches"]:
                break
            image, target_norm = batch  # full images/targets stay on CPU
            target = normalizer.decode_temperature(target_norm)
            pred = normalizer.decode_temperature(predict_field(image))
            check_finite("evaluation.prediction", pred, {"batch": batch_idx})
            error_stats.update(pred, target)
            batch_size = image.shape[0]
            total_rel_l2 += relative_l2_per_sample(pred, target).sum().item()
            total_mae += mae_metric(pred, target).item() * batch_size
            total_samples += batch_size
        if not total_samples:
            raise ValueError("No evaluation batches selected")
        metrics = {"rel_l2": total_rel_l2 / total_samples, "mae": total_mae / total_samples}
        metrics.update(error_stats.result())
        return metrics

## 10. Run training

In [ ]:
def format_error_stats(label, metrics):
    return (f"{label} RelL2 distribution: Max {metrics['max_rel_l2']:.6f}, "
            f"Median {metrics['median_rel_l2']:.6f}, P90 {metrics['p90_rel_l2']:.6f}, "
            f"P95 {metrics['p95_rel_l2']:.6f}, P99 {metrics['p99_rel_l2']:.6f}")

training_failed = False
failure_message = None
best_val = float("inf")
try:
    for epoch in range(1, HPARAMS["epochs"] + 1):
        train_metrics = train_one_epoch_debug(epoch)
        val_metrics = evaluate_debug(val_loader)
        # scheduler.step()
        for split, metrics in (("train", train_metrics), ("val", val_metrics)):
            for metric, value in metrics.items():
                history[f"{split}_{metric}"].append(value)
        history["lr"].append(optimizer.param_groups[0]["lr"])
        checkpoint = {"model_state_dict": model.state_dict(), "model_kwargs": model_kwargs,
                      "package_versions": PACKAGE_VERSIONS,
                      "optimizer_state_dict": optimizer.state_dict(),
                      "scheduler_state_dict": scheduler.state_dict(), "epoch": epoch,
                      "normalization": normalizer.to_dict(), "hparams": HPARAMS,
                      "history": history, "val_metrics": val_metrics}
        torch.save(checkpoint, HPARAMS["last_checkpoint_path"])
        if val_metrics["rel_l2"] < best_val:
            best_val = val_metrics["rel_l2"]
            torch.save(checkpoint, HPARAMS["checkpoint_path"])
        print(f"\nEpoch {epoch:03d}/{HPARAMS['epochs']} | "
              f"Train RelL2: {train_metrics['rel_l2']:.6f} | Val RelL2: {val_metrics['rel_l2']:.6f} | "
              f"Train MAE: {train_metrics['mae']:.6f} | Val MAE: {val_metrics['mae']:.6f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.3e} | "
              f"{format_error_stats('Train', train_metrics)} | "
              f"{format_error_stats('Val', val_metrics)}\n")
except (FloatingPointError, RuntimeError) as exc:
    training_failed = True
    failure_message = str(exc)
    print("TRAINING STOPPED:", failure_message)
    raise
finally:
    diagnostic_context = None
    step_df = pd.DataFrame(step_log)
    step_df.to_csv(HPARAMS["debug_csv_path"], index=False)
    print("Debug CSV:", HPARAMS["debug_csv_path"])
print("training_failed:", training_failed)

## 11. Core plots

In [ ]:
def plot_series(df, columns, ylabel, title, logy=False):
    if df is None or len(df) == 0:
        print("No data for:", title)
        return

    plt.figure(figsize=(9, 5))

    for col in columns:
        if col in df.columns:
            plt.plot(
                df["global_optimizer_step"],
                df[col],
                label=col,
            )

    plt.xlabel("Optimizer step")
    plt.ylabel(ylabel)
    plt.title(title)

    if logy:
        plt.yscale("log")

    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


if len(step_log):
    step_df = pd.DataFrame(step_log)

    plot_series(
        step_df,
        ["loss_rel_l2"],
        "Relative L2",
        "Training loss",
        logy=True,
    )

    plot_series(
        step_df,
        ["raw_grad_norm"],
        "Gradient norm before clipping",
        "Raw gradient norm",
        logy=True,
    )



## 12. Gradients and parameter values

In [ ]:
if len(step_log):
    # Gradient norms
    plot_series(
        step_df,
        [
            "grad_input_encoder_norm",
            "grad_blocks_norm",
            "grad_decoder_norm",
        ],
        "Gradient L2 norm",
        "Gradient norm by network section",
        logy=True,
    )

    # Gradient maximums
    plot_series(
        step_df,
        [
            "grad_input_encoder_absmax",
            "grad_blocks_absmax",
            "grad_decoder_absmax",
        ],
        "Max |gradient|",
        "Maximum absolute gradient by network section",
        logy=True,
    )

    # Parameter norms
    plot_series(
        step_df,
        [
            "param_input_encoder_norm",
            "param_blocks_norm",
            "param_decoder_norm",
        ],
        "Parameter L2 norm",
        "Parameter norm by network section",
        logy=True,
    )

    # Parameter maximums
    plot_series(
        step_df,
        [
            "param_input_encoder_absmax",
            "param_blocks_absmax",
            "param_decoder_absmax",
        ],
        "Max |parameter|",
        "Maximum absolute parameter by network section",
        logy=True,
    )

In [ ]:
final_parameter_snapshot = snapshot_parameter_values(model)

for group_name, final_values in final_parameter_snapshot.items():
    initial_values = initial_parameter_snapshot.get(
        group_name,
        np.array([]),
    )

    plt.figure(figsize=(8, 4))

    if len(initial_values):
        plt.hist(
            initial_values,
            bins=80,
            alpha=0.5,
            label="initial",
        )

    if len(final_values):
        plt.hist(
            final_values,
            bins=80,
            alpha=0.5,
            label="current/final",
        )

    plt.xlabel("Parameter value")
    plt.ylabel("Count")
    plt.title(f"Parameter distribution — {group_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


if last_gradient_snapshot is not None:
    for group_name, values in last_gradient_snapshot.items():
        if len(values) == 0:
            continue

        plt.figure(figsize=(8, 4))
        plt.hist(values, bins=100)
        plt.xlabel("Gradient value")
        plt.ylabel("Count")
        plt.title(f"Latest gradient distribution — {group_name}")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

## 13. U-Net encoder and decoder internals

In [ ]:
internal_df = recorder.dataframe()
if len(internal_df):
    tensor_df = internal_df[internal_df["kind"] == "tensor"].copy()
    for metric in ("rms", "absmax"):
        plt.figure(figsize=(9, 4))
        for tag in tensor_df["tag"].unique():
            d = tensor_df[tensor_df["tag"] == tag].reset_index(drop=True)
            plt.plot(np.arange(len(d)), d[metric], label=tag)
        plt.xlabel("Logged forward pass"); plt.ylabel(metric)
        plt.title("Encoder/decoder activations: " + metric)
        plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
else:
    print("No detailed internal records.")


## 14. Layer output evolution

In [ ]:
internal_df = recorder.dataframe()
tensor_df = internal_df[internal_df["kind"] == "tensor"].copy()
if len(tensor_df):
    latest_epoch = tensor_df["epoch"].max()
    d = tensor_df[tensor_df["epoch"] == latest_epoch]
    d = d[d["batch"] == d["batch"].max()]
    display(d[["epoch", "batch", "tag", "mean", "std", "rms", "absmax", "norm"]])
    for metric in ("rms", "absmax"):
        plt.figure(figsize=(9, 5))
        plt.plot(np.arange(len(d)), d[metric], marker="o")
        plt.xlabel("Encoder/decoder stage"); plt.ylabel(metric)
        plt.title("Layer output " + metric); plt.grid(True); plt.tight_layout(); plt.show()

## 15. Epoch curves

In [ ]:
if len(history["train_rel_l2"]):
    e = np.arange(1, len(history["train_rel_l2"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(e, history["train_rel_l2"], label="Train")
    plt.plot(e, history["val_rel_l2"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Relative L2")
    plt.yscale("log")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(e, history["train_mae"], label="Train")
    plt.plot(e, history["val_mae"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("MAE")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No complete epoch recorded.")

## 16. Interpretation

This is a **regression adaptation using nnU-Net's packaged 2D PlainConvUNet**, instantiated through `nnunetv2.utilities.get_network_from_plans`. The architecture code is installed with pip and used without modification. We manually configure its stages; this is not the full self-configuring nnU-Net segmentation workflow. There is no automatic planner, segmentation preprocessing, Dice/cross-entropy loss, label quantization, softmax, argmax, or segmentation postprocessing.

`pip install nnunetv2==2.7.0 dynamic-network-architectures==0.4.3` installs the tested external versions. The small `nn_unet.py` adapter builds the official network with eight input channels, one linear temperature output and deep supervision disabled. Arbitrary grid sizes are padded to the network divisor, then cropped back. The existing training loop uses relative L2 on decoded Celsius values, with MAE for evaluation.

The point dataset is reshaped into `[8,H,W]`: x/y coordinates, standardized power, chip occupancy, ceramic/baseplate/copper masks, and h/10000. Targets remain continuous standardized temperature maps `[1,H,W]`. Normalization uses all selected training files and is cached. Training uses the same seed-42 5% subset; validation remains complete and held-out testing is disabled by default (`run_test=False`). No target-derived `max_temps` are inputs. Existing official splits can contain overlapping layouts, so results do not establish unseen-layout generalization.

The nnU-Net pipeline itself is built for segmentation. Its pretrained segmentation weights are not used here; the U-Net starts from scratch. Diagnostics record encoder/decoder activations, parameter distributions and gradients. There are no attention slices in this model.

Sources: [nnU-Net](https://github.com/MIC-DKFZ/nnUNet), [official network factory](https://github.com/MIC-DKFZ/nnUNet/blob/master/nnunetv2/utilities/get_network_from_plans.py). Cite Isensee et al., *nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation*, Nature Methods (2021), when using the package.

Dataset: [Multi-topology Dataset](https://github.com/henkuailederen/Multi-topology-Dataset), Ruiting Ke, Jianfeng Tao and Chengliang Liu, CC BY 4.0. Cite the repository and accompanying study when using these data.


**Sliding-window variant:** training uses random 64×64 crops without augmentation, while validation reconstructs full boards. The dry run and validation use overlapping 64×64 tiles, nnU-Net's `compute_steps_for_sliding_window`, and its `compute_gaussian` importance weights. Tile predictions are accumulated in float32 on CPU and divided by the total weight at every pixel. Only tile batches are transferred to the GPU. The original full-image coordinates, masks, cooling and normalization are retained in each crop. Gaussian blending reduces the contribution of patch edges; it does not guarantee predictions equal full-image inference, especially with InstanceNorm or long-range thermal effects. Set the patch size to `[256,256]` for a single tile on native-size data. No second network or coarse-to-fine cascade is used.

Held-out testing remains disabled; when enabled, its metrics and plots use the same tiled predictor. Best-checkpoint selection also uses sliding-window validation. Checkpoints and outputs use separate sliding-window filenames/directories. Source: [nnU-Net sliding-window utilities](https://github.com/MIC-DKFZ/nnUNet/blob/master/nnunetv2/inference/sliding_window_prediction.py).


Both training and validation epoch summaries include exact maximum, median (P50), P90, P95 and P99 absolute pixel errors in Celsius. Training metrics use the sampled-patch predictions during optimizer updates. Validation metrics use the final blended full-image sliding-window prediction, counting each original pixel once rather than counting overlapping tiles multiple times. Exact quantiles use temporary disk-backed arrays which are cleaned up after each split. Summaries are stored in checkpoint history. Training samples cases with replacement and uniformly samples aligned crop locations; there is no augmentation or foreground-biased sampling.


**Random patch training:** each epoch contains `train_iterations_per_epoch` batches of `batch_size` patches (currently 20 x 32 = 640 sampled patches). Cases are sampled with replacement from the fixed seed-42 5% training-image subset. Each patch is 64×64, with fresh crop locations each epoch. All eight input channels and the temperature target use the same crop. Original board coordinates, cooling and training-set normalization are preserved. There are no rotations, flips, intensity changes, target quantization, or foreground oversampling. Sampling is reproducible from seed, epoch and sample index, independent of worker ordering. The regression sampler follows nnU-Net's random-crop idea; it does not invoke the native segmentation data loader. nnU-Net defaults to 250 training batches; our current iteration count and patch size are explicit user settings. Sources: [official loader](https://github.com/MIC-DKFZ/nnUNet/blob/v2.7.0/nnunetv2/training/dataloading/data_loader.py), [trainer defaults](https://github.com/MIC-DKFZ/nnUNet/blob/v2.7.0/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py#L139).

Training Max/Median/P90/P95/P99 summarize the per-patch RelL2 ratios from sampled training patches. Validation summaries use one RelL2 ratio per reconstructed full board. Full validation runs after each epoch; testing remains disabled. The full-image notebook uses full-image training ratios. Smoke tests reduce training to two batches per epoch.

**Kaggle setup:** enable Internet and a GPU; attach the dataset at the explicit `DATA_ROOT` shown in cell 0. The notebook downloads only four Python modules from our repository and installs external packages if needed. It reads sample names directly from `metadata/manifest.csv`; it never searches Kaggle input directories or recursively discovers samples. Checkpoints, normalization statistics and logs are written under `/kaggle/working/nn_unet_pcb_sliding_window`. If the imported dataset has an extra wrapper directory, edit DATA_ROOT explicitly.


## 17. Best checkpoint and held-out test predictions

In [ ]:
checkpoint = torch.load(HPARAMS["checkpoint_path"], map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
normalizer = NormalizationStats(**checkpoint["normalization"])
print("Best epoch:", checkpoint["epoch"])
if HPARAMS["run_test"]:
    test_metrics = evaluate_debug(test_loader)
    print("Test Relative L2:", test_metrics["rel_l2"])
    print("Test MAE (C):", test_metrics["mae"])
    (WORKING_DIR / "nn_unet_pcb_sliding_window_test_metrics.json").write_text(json.dumps(test_metrics, indent=2))

    model.eval()
    image, target_norm = test_dataset[0]
    with torch.no_grad():
        pred_norm = predict_field(image[None])[0]
    height = image.shape[-2]
    true_c = normalizer.decode_temperature(target_norm).numpy().reshape(height, -1)
    pred_c = normalizer.decode_temperature(pred_norm).numpy().reshape(true_c.shape)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    lo, hi = min(true_c.min(), pred_c.min()), max(true_c.max(), pred_c.max())
    for ax, values, title in zip(axes, [true_c, pred_c, np.abs(pred_c - true_c)],
                                ["Reference (C)", "Prediction (C)", "Absolute error (C)"]):
        im = ax.imshow(values, **({} if "error" in title else dict(vmin=lo, vmax=hi)))
        ax.set_title(title); ax.axis("off"); fig.colorbar(im, ax=ax, shrink=0.75)
    plt.tight_layout(); plt.savefig(WORKING_DIR / "nn_unet_pcb_sliding_window_test_prediction.png", dpi=140); plt.show()
    np.savez_compressed(WORKING_DIR / "nn_unet_pcb_sliding_window_test_prediction.npz", reference_c=true_c, prediction_c=pred_c)
else:
    print("Held-out testing skipped (run_test=False).")
print("Best checkpoint:", HPARAMS["checkpoint_path"])
for handle in hook_handles:
    handle.remove()